`conda activate r_python`

In [13]:
library(tidyverse)
library(Seurat)
library(stringr)
library(tibble)
library(data.table)
library(Matrix)
library(dplyr)
library(future)
library(HGNChelper)
library(BiocParallel)
library(scDblFinder)
library(DoubletFinder)
library(KernSmooth)
library(cluster)
library(parallel)
library(dittoSeq)
library(scCustomize)
library(patchwork)
source("scripts/helper_functions_v1.R")
set.seed(1)
plan("multiprocess", workers = 24)
options(future.globals.maxSize = 1000 * 1024^6)

── Attaching core tidyverse packages ──────────────
✔ dplyr     1.1.0     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.1     ✔ tibble    3.1.8
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.1     
── Conflicts ───────────── tidyverse_conflicts() ──
✖ lubridate::%within%() masks IRanges::%within%()
✖ dplyr::collapse()     masks IRanges::collapse()
✖ dplyr::combine()      masks Biobase::combine(), BiocGenerics::combine()
✖ dplyr::desc()         masks IRanges::desc()
✖ tidyr::expand()       masks S4Vectors::expand()
✖ dplyr::filter()       masks stats::filter()
✖ dplyr::first()        masks S4Vectors::first()
✖ dplyr::lag()          masks stats::lag()
✖ ggplot2::Position()   masks BiocGenerics::Position(), base::Position()
✖ purrr::reduce()       masks GenomicRanges::reduce(), IRanges::reduce()
✖ dplyr::rename()       masks S4Vectors::rename()
✖ lubridate::second()   masks S4Vectors::second()
✖ lubridate::second<-() masks S4Vectors::second<-()
✖ dplyr

In [2]:
meta_cohort <- readxl::read_excel('../data/dmg_atlas_metadata.xlsx',
                                  sheet = "Metadata")
meta_cohort

Study,Institute,Preservation_method,ID,Diagnosis,Tumor_type,Tumor_subtype,Location,Source,Clinical_status,⋯,PPM1D,ACVR1,TSHR,BRAF,GNAQ,LMNA,KIT,ARID1A,KRAS,Other
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Ruiz2023,PMC,Cryo,T18-90532,DMG Pons,DMG H3 K27-altered,H3.1 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T19-90171,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Autopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T19-91014,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-01237,DMG Spinal,DMG H3 K27-altered,H3.3 K27-mutant,Spinal,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,NA,T20-90151,DMG Thalamus,DMG H3 K27-altered,H3.3 K27-mutant,Thalamus,Biopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-90239,DMG Spinal,DMG H3 K27-altered,H3.3 K27-mutant,Spinal,Biopsy,Recurrence,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A
Ruiz2023,PMC,Cryo,T20-90296,DMG Pons,DMG H3 K27-altered,H3.2 K27-mutant,Pons,Biopsy,Primary/Recurrence,⋯,WT,M,WT,WT,WT,WT,WT,WT,WT,FGFR3 overexpression; TP53 inactivation
Ruiz2023,PMC,Cryo,T20-93623,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,RGMA-ERAS fusion
Ruiz2023,PMC,Cryo/Snap-frozen,T20-93210,DMG Pons,DMG H3 K27-altered,H3.3 K27-mutant,Pons,Biopsy,Primary,⋯,WT,WT,WT,WT,WT,WT,WT,WT,WT,N/A


### Filbin2018

DOI: 10.1126/science.aao4750

##### Read TPM normalized matrices

In [3]:
counts <- as.matrix(fread(paste0('/hpc/pmc_stunnenberg/cruiz/scRNA/',
             'markers-and-databases/dmg/Filbin2018/',
             'K27Mproject.RSEM.vh20170621.txt'),
                             blank.lines.skip=FALSE) %>% column_to_rownames('Gene'))

gene.names <- checkGeneSymbols(rownames(counts), unmapped.as.na=FALSE)
gene.names
rownames(counts) <- make.unique(gene.names$Suggested.Symbol)

counts

Maps last updated on: Thu Oct 24 12:31:05 2019

Warning message in checkGeneSymbols(rownames(counts), unmapped.as.na = FALSE):
“Human gene symbols should be all upper-case except for the 'orf' in open reading frames. The case of some letters was corrected.”
Warning message in checkGeneSymbols(rownames(counts), unmapped.as.na = FALSE):
“x contains non-approved gene symbols”


x,Approved,Suggested.Symbol
<chr>,<lgl>,<chr>
A1BG,TRUE,A1BG
A1BG-AS1,TRUE,A1BG-AS1
A1CF,TRUE,A1CF
A2M,TRUE,A2M
A2M-AS1,TRUE,A2M-AS1
A2ML1,TRUE,A2ML1
A2MP1,TRUE,A2MP1
A4GALT,TRUE,A4GALT
A4GNT,TRUE,A4GNT


,MUV1-P04-B12,MUV1-P04-C08,MUV1-P04-D09,MUV1-P04-D10,MUV1-P04-E03,MUV1-P04-E07,MUV1-P04-E08,MUV1-P04-E10,MUV1-P04-E11,MUV1-P04-F05,⋯,Oligo-P22-G05,Oligo-P22-G07,Oligo-P22-G11,Oligo-P22-G12,Oligo-P22-H02,Oligo-P22-H03,Oligo-P22-H05,Oligo-P22-H06,Oligo-P22-H08,Oligo-P22-H09
A1BG,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,132.26,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A1BG-AS1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A1CF,0.00,0.00,0.00,0.00,0.00,0.00,0.53,0.34,0.00,0.00,⋯,0.00,0.65,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A2M,0.00,0.00,0.00,0.00,348.48,362.08,0.00,0.00,0.00,27.17,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A2M-AS1,0.00,0.00,0.00,0.00,0.00,1.19,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1098.07,0.00
A2ML1,0.00,0.92,0.00,0.00,1.25,0.33,0.00,0.85,4.19,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,5.88,0.00,0.00,0.00
A2MP1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A4GALT,0.00,0.00,0.00,0.00,0.00,29.25,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
A4GNT,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
AA06,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [4]:
Filbin2018 <- CreateSeuratObject(counts, min.features = 0, min.cells = 0, 
                                  project = 'Filbin2018')
Filbin2018$ID <- gsub("-.*","\\1",colnames(Filbin2018)) # https://datascience.stackexchange.com/questions/8922/removing-strings-after-a-certain-character-in-a-given-text
Filbin2018@meta.data

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


,orig.ident,nCount_RNA,nFeature_RNA,ID
,<fct>,<dbl>,<int>,<chr>
MUV1-P04-B12,Filbin2018,1000000.3,4811,MUV1
MUV1-P04-C08,Filbin2018,1000000.3,2617,MUV1
MUV1-P04-D09,Filbin2018,1000000.0,4628,MUV1
MUV1-P04-D10,Filbin2018,1000000.1,4882,MUV1
MUV1-P04-E03,Filbin2018,1000000.4,5927,MUV1
MUV1-P04-E07,Filbin2018,1000000.3,7844,MUV1
MUV1-P04-E08,Filbin2018,1000000.4,5814,MUV1
MUV1-P04-E10,Filbin2018,999999.8,5833,MUV1
MUV1-P04-E11,Filbin2018,1000000.4,5992,MUV1


In [5]:
table(Filbin2018$ID)


      BCH1126        BCH836        BCH869 BCH869_DGC100  BCH869_DGC75 
          299           527           492            89           165 
    BCH869_GF     BCH869_GS    BCH869_PDX     BCH869_SF        MGH101 
           68           290           167            84            92 
       MGH104         MGH66          MUV1         MUV10          MUV5 
           65           442           146           286           708 
        Oligo 
          138 

In [6]:
# There are three sample from this study that were also included later in Liu2022
# Checking the barcodes there was 444 matches between both datsets that couls also be due to the method the use (plate-based)
# and the Liu2022 methods do not mention they integrated cells from the previous study
# All cells will be kept

Filbin2018 <- subset(Filbin2018, subset = ID %in% (meta_cohort %>% 
                     filter(Study %in% c('Filbin2018', 'Liu2022')) %>% pull(ID)))
Filbin2018

An object of class Seurat 
23686 features across 2458 samples within 1 assay 
Active assay: RNA (23686 features, 0 variable features)

In [7]:
Filbin2018_meta <- as.data.frame(fread(paste0('/hpc/pmc_stunnenberg/cruiz/scRNA/',
             'markers-and-databases/dmg/Filbin2018/',
             'PortalK27M_Metadata.vh20180223.txt'))) 
Filbin2018_meta <- Filbin2018_meta[-1,]
Filbin2018_meta <- Filbin2018_meta %>% filter(NAME %in% colnames(Filbin2018)) %>% 
                      column_to_rownames('NAME')

Filbin2018_meta

,Sample,GenesExpressed,HousekeepingGeneExpression,Type,Cellcycle,OPC-variable,OC-like,AC-like,OPC-like
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
BCH1126-P01-A01,BCH1126,4064,6.371650316,Malignant,-0.44186567,-1.018729626,-0.511928109,-0.523686956,0.83940171
BCH1126-P01-A02,BCH1126,5162,6.363841855,Malignant,-0.490522093,-0.938585898,-0.018924871,-1.02391073,0.400311983
BCH1126-P01-A04,BCH1126,3583,4.623983418,Malignant,0.319448462,-0.870724574,0.327553519,0.303905206,-0.314469209
BCH1126-P01-A07,BCH1126,4743,6.131107408,Malignant,-0.747714419,-0.444016923,-0.461922304,-0.421771418,1.11865115
BCH1126-P01-A08,BCH1126,4015,5.803742692,Malignant,-0.219426632,-0.074824661,0.304706623,0.184920455,1.718093353
BCH1126-P01-A09,BCH1126,5742,6.430943464,Malignant,-0.577234556,-1.3868627,0.329421293,-0.88265844,-0.013435158
BCH1126-P01-B01,BCH1126,3044,4.32654854,Immune cell,NA,NA,NA,NA,NA
BCH1126-P01-B03,BCH1126,4889,6.075816313,Malignant,-0.46423596,-0.118795718,0.257534493,-0.387274608,1.493154986
BCH1126-P01-B04,BCH1126,5565,6.054379026,Malignant,-0.629568899,-0.893449316,0.894925577,-1.70173949,0.228877493


In [8]:
table(Filbin2018_meta$Type)


         Filter     Immune cell       Malignant Oligodendrocyte 
              9              96            2259              94 

##### Check final metadata

In [10]:
# Metadata of MUV samples was more complete in Liu2022 and we kept those. Survivael info was not present in Liu2022
# Added survival from Filbin2018 MUV samples to Liu2022

Filbin2018 <- AddMetaData(Filbin2018, Filbin2018_meta %>% select(Type))
Filbin2018$Original_annotation <- Filbin2018$Type
Filbin2018$SampleID <- Filbin2018$ID
Filbin2018$Data <- 'RNA_only'
Filbin2018$Batch_for_correction <- 'SmartSeq2_cell_rna'

Filbin2018$Type <- NULL
Filbin2018$orig.ident <- NULL

Filbin2018@meta.data

,nCount_RNA,nFeature_RNA,ID,Original_annotation,SampleID,Data,Batch_for_correction
,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
MUV1-P04-B12,1000000.3,4811,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-C08,1000000.3,2617,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-D09,1000000.0,4628,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-D10,1000000.1,4882,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-E03,1000000.4,5927,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-E07,1000000.3,7844,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-E08,1000000.4,5814,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-E10,999999.8,5833,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna
MUV1-P04-E11,1000000.4,5992,MUV1,Malignant,MUV1,RNA_only,SmartSeq2_cell_rna


In [11]:
saveRDS(Filbin2018, '../data/Filbin2018_no_filtered.rds')

### Liu2022

DOI:10.1038/s41588-022-01236-3

In [4]:
path.dir <- '/hpc/pmc_stunnenberg/cruiz/scRNA/markers-and-databases/dmg/Liu2022/'

In [5]:
sc_mtx <- fread(paste0(path.dir, 'GSE184357_counts_fresh_preFilter.txt.gz')) %>%
            column_to_rownames('V1')
gene.names <- checkGeneSymbols(rownames(sc_mtx), unmapped.as.na=FALSE)
gene.names
rownames(sc_mtx) <- make.unique(gene.names$Suggested.Symbol)
sc_mtx

Warning message in fread(paste0(path.dir, "GSE184357_counts_fresh_preFilter.txt.gz")):
“Detected 8251 column names but the data has 8252 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”
Maps last updated on: Thu Oct 24 12:31:05 2019

Warning message in checkGeneSymbols(rownames(sc_mtx), unmapped.as.na = FALSE):
“Human gene symbols should be all upper-case except for the 'orf' in open reading frames. The case of some letters was corrected.”
Warning message in checkGeneSymbols(rownames(sc_mtx), unmapped.as.na = FALSE):
“x contains non-approved gene symbols”


x,Approved,Suggested.Symbol
<chr>,<lgl>,<chr>
5S_rRNA,FALSE,5S_rRNA
7SK,FALSE,RN7SK
A1BG,TRUE,A1BG
A1BG-AS1,TRUE,A1BG-AS1
A1CF,TRUE,A1CF
A2M,TRUE,A2M
A2M-AS1,TRUE,A2M-AS1
A2ML1,TRUE,A2ML1
A2ML1-AS1,TRUE,A2ML1-AS1


,BT1126.P1.A01,BT1126.P1.A02,BT1126.P1.A03,BT1126.P1.A04,BT1126.P1.A05,BT1126.P1.A06,BT1126.P1.A07,BT1126.P1.A08,BT1126.P1.A09,BT1126.P1.A10,⋯,MUV87.P2.H03,MUV87.P2.H04,MUV87.P2.H05,MUV87.P2.H06,MUV87.P2.H07,MUV87.P2.H08,MUV87.P2.H09,MUV87.P2.H10,MUV87.P2.H11,MUV87.P2.H12
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5S_rRNA,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
RN7SK,5.68,0.00,0.00,0.00,0.00,0,0.00,0.00,12.91,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
A1BG,0.00,73.49,0.00,0.00,1.41,0,1.04,0.00,56.86,0.00,⋯,19.90,4.20,54.59,0.00,0.00,2.23,32.96,0.00,0.00,0
A1BG-AS1,0.00,0.00,0.00,0.68,0.00,0,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,32.64,0.00,0.00,0.00,0.00,0.00,0
A1CF,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
A2M,0.00,0.00,0.00,0.00,2437.14,0,0.00,0.00,55.30,0.00,⋯,0.00,0.00,0.00,34.57,0.00,137.88,11.89,173.49,0.00,0
A2M-AS1,0.00,0.00,0.00,0.00,1.44,0,0.00,0.00,117.03,0.00,⋯,0.00,36.83,0.00,94.17,0.00,0.00,0.00,0.00,0.00,0
A2ML1,0.00,0.00,0.00,0.00,1.04,0,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,1.96,0.92,0.00,0.00,0.00,0
A2ML1-AS1,0.00,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [6]:
sn_mtx <- fread(paste0(path.dir, 'GSE184357_counts_frozen_preFilter.txt.gz')) %>%
            column_to_rownames('V1')
gene.names <- checkGeneSymbols(rownames(sn_mtx), unmapped.as.na=FALSE)
gene.names
rownames(sn_mtx) <- make.unique(gene.names$Suggested.Symbol)
sn_mtx

Warning message in fread(paste0(path.dir, "GSE184357_counts_frozen_preFilter.txt.gz")):
“Detected 10176 column names but the data has 10177 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”
Maps last updated on: Thu Oct 24 12:31:05 2019

Warning message in checkGeneSymbols(rownames(sn_mtx), unmapped.as.na = FALSE):
“Human gene symbols should be all upper-case except for the 'orf' in open reading frames. The case of some letters was corrected.”
Warning message in checkGeneSymbols(rownames(sn_mtx), unmapped.as.na = FALSE):
“x contains non-approved gene symbols”


x,Approved,Suggested.Symbol
<chr>,<lgl>,<chr>
5S_rRNA,FALSE,5S_rRNA
7SK,FALSE,RN7SK
A1BG,TRUE,A1BG
A1BG-AS1,TRUE,A1BG-AS1
A1CF,TRUE,A1CF
A2M,TRUE,A2M
A2M-AS1,TRUE,A2M-AS1
A2ML1,TRUE,A2ML1
A2ML1-AS1,TRUE,A2ML1-AS1


,MUV82.P1.A01,MUV82.P1.A02,MUV82.P1.A03,MUV82.P1.A04,MUV82.P1.A05,MUV82.P1.A06,MUV82.P1.A07,MUV82.P1.A08,MUV82.P1.A09,MUV82.P1.A10,⋯,S1D2E3H7.P2.H03,S1D2E3H7.P2.H04,S1D2E3H7.P2.H05,S1D2E3H7.P2.H06,S1D2E3H7.P2.H07,S1D2E3H7.P2.H08,S1D2E3H7.P2.H09,S1D2E3H7.P2.H10,S1D2E3H7.P2.H11,S1D2E3H7.P2.H12
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5S_rRNA,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,0.00
RN7SK,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.40,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,38.05
A1BG,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1290.30,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,0.00
A1BG-AS1,24.28,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,3.12
A1CF,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.71,0.00,⋯,0.00,0.77,0.0,0,1.72,0.00,0.00,0,0.00,0.31
A2M,67.47,237.88,111.46,0.00,2715.89,0.00,0.00,62.45,81.05,0.29,⋯,0.00,0.00,7445.1,0,0.00,0.00,3.78,0,0.00,0.97
A2M-AS1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,0.00
A2ML1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,5.69,0,0.00,70.12
A2ML1-AS1,1.16,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.00,0.00,0.0,0,0.00,0.00,0.00,0,0.00,2.48


In [7]:
sc_meta <- fread(paste0(path.dir, 'GSE184357_metadata_fresh_postFilter.txt.gz')) %>%
            column_to_rownames('cell')  %>% mutate(Isolation_method_by_cell = 'Cell')
sc_meta

,sample,age,location,annotation,Isolation_method_by_cell
,<chr>,<chr>,<chr>,<chr>,<chr>
BT1126.P1.A01,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.A02,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.A04,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.A07,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.A08,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.A09,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.B01,BT1126,pediatric,pontine,Microglia,Cell
BT1126.P1.B03,BT1126,pediatric,pontine,OPC-like-2,Cell
BT1126.P1.B04,BT1126,pediatric,pontine,OPC-like-2,Cell


In [8]:
sn_meta <- fread(paste0(path.dir, 'GSE184357_metadata_frozen_postFilter.txt.gz')) %>%
            column_to_rownames('cell') %>% mutate(Isolation_method_by_cell = 'Nuclei')
sn_meta

,sample,age,location,annotation,Isolation_method_by_cell
,<chr>,<chr>,<chr>,<chr>,<chr>
MUV82.P1.A02,MUV82,adult,thalamic,Myeloid,Nuclei
MUV82.P1.A05,MUV82,adult,thalamic,Myeloid,Nuclei
MUV82.P1.A06,MUV82,adult,thalamic,Oligodendrocyte,Nuclei
MUV82.P1.A07,MUV82,adult,thalamic,T cell,Nuclei
MUV82.P1.A08,MUV82,adult,thalamic,T cell,Nuclei
MUV82.P1.A09,MUV82,adult,thalamic,Myeloid,Nuclei
MUV82.P1.A10,MUV82,adult,thalamic,OPC-like-1,Nuclei
MUV82.P1.A11,MUV82,adult,thalamic,Myeloid,Nuclei
MUV82.P1.B01,MUV82,adult,thalamic,MES-like,Nuclei


In [9]:
sc <-  CreateSeuratObject(sc_mtx, min.features = 0, min.cells = 0,
                          meta.data = sc_meta,
                          project = 'sc_Liu2022')
sc$ID <- gsub("\\..*","\\1",colnames(sc))
sc$Batch_for_correction <- 'SmartSeq2_cell_rna'

sn <-  CreateSeuratObject(sn_mtx, min.features = 0, min.cells = 0,
                          meta.data = sn_meta,
                          project = 'sn_Liu2022')
sn$Batch_for_correction <- 'SmartSeq2_nuclei_rna'
sn$ID <- gsub("\\..*","\\1",colnames(sn))
sc
sn

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


An object of class Seurat 
32617 features across 8251 samples within 1 assay 
Active assay: RNA (32617 features, 0 variable features)

An object of class Seurat 
32617 features across 10176 samples within 1 assay 
Active assay: RNA (32617 features, 0 variable features)

In [10]:
Liu2022 <- merge(sc, sn,
                 add.cell.ids = c('sc', 'sn'), # Some cell names are duplicated across objects provided
                 #avoid it by adding source
                 project = "Liu2022", 
                 merge.data = FALSE)
Liu2022$Original_annotation <- Liu2022$annotation
Liu2022$SampleID <- Liu2022$ID
Liu2022$Data <- 'RNA_only'
Liu2022
str(Liu2022)

An object of class Seurat 
32617 features across 18427 samples within 1 assay 
Active assay: RNA (32617 features, 0 variable features)

Formal class 'Seurat' [package "SeuratObject"] with 13 slots
  ..@ assays      :List of 1
  .. ..$ RNA:Formal class 'Assay' [package "SeuratObject"] with 8 slots
  .. .. .. ..@ counts       :Formal class 'dgCMatrix' [package "Matrix"] with 6 slots
  .. .. .. .. .. ..@ i       : int [1:92979757] 1 13 14 20 22 24 29 30 33 38 ...
  .. .. .. .. .. ..@ p       : int [1:18428] 0 5364 11213 13818 19224 24676 25274 31102 36453 43284 ...
  .. .. .. .. .. ..@ Dim     : int [1:2] 32617 18427
  .. .. .. .. .. ..@ Dimnames:List of 2
  .. .. .. .. .. .. ..$ : chr [1:32617] "5S-rRNA" "RN7SK" "A1BG" "A1BG-AS1" ...
  .. .. .. .. .. .. ..$ : chr [1:18427] "sc_BT1126.P1.A01" "sc_BT1126.P1.A02" "sc_BT1126.P1.A03" "sc_BT1126.P1.A04" ...
  .. .. .. .. .. ..@ x       : num [1:92979757] 5.68 249.4 0.31 0.36 1.36 ...
  .. .. .. .. .. ..@ factors : list()
  .. .. .. ..@ data         :Formal class 'dgCMatrix' [package "Matrix"] with 6 slots
  .. .. .. .. .. ..@ i       : int [1:92979757] 1 13 14 20 22 24 29 30 3

In [11]:
Liu2022$orig.ident <- NULL
Liu2022$sample <- NULL
Liu2022$age <- NULL
Liu2022$location <- NULL
Liu2022$annotation <- NULL

In [12]:
# Since we will only include pediatric patients (<21yo), the meatadata provided only contains ID for samples that meet that criteria  
# We filter out samples from older patients
Liu2022 <- subset(Liu2022, subset = ID %in% (meta_cohort %>% 
                     filter(Study == 'Liu2022') %>% pull(ID)))
Liu2022

An object of class Seurat 
32617 features across 13823 samples within 1 assay 
Active assay: RNA (32617 features, 0 variable features)

In [13]:
Liu2022@meta.data

,nCount_RNA,nFeature_RNA,Isolation_method_by_cell,ID,Batch_for_correction,Original_annotation,SampleID,Data
,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
sc_BT1126.P1.A01,1000000.0,5364,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only
sc_BT1126.P1.A02,1000000.1,5849,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only
sc_BT1126.P1.A03,1000000.0,2605,NA,BT1126,SmartSeq2_cell_rna,NA,BT1126,RNA_only
sc_BT1126.P1.A04,1000000.1,5406,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only
sc_BT1126.P1.A05,1000000.2,5452,NA,BT1126,SmartSeq2_cell_rna,NA,BT1126,RNA_only
sc_BT1126.P1.A06,1000000.0,598,NA,BT1126,SmartSeq2_cell_rna,NA,BT1126,RNA_only
sc_BT1126.P1.A07,1000000.0,5828,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only
sc_BT1126.P1.A08,1000000.1,5351,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only
sc_BT1126.P1.A09,1000000.0,6831,Cell,BT1126,SmartSeq2_cell_rna,OPC-like-2,BT1126,RNA_only


In [14]:
saveRDS(Liu2022, '../data/Liu2022_no_filtered.rds')

Notes: <br>
- Sample AAA07653 has a typo in table S1. In the cell metadata is AAA007653 (an additional zero)
- Sample MUV7 is in Table S1 but it is not shown in any figure of the manuscript. We assume it was not included in the final paper.
- All the others were only profiled by in situ sequencing 